# 백트래킹 (Backtracking)

`-` 해를 탐색하는 도중 해가 아니면 되돌아가서 다시 해를 탐색하는 알고리즘

## 좋은수열

- 문제 출처: [정올 1027번](https://jungol.co.kr/problem/1027)

`-` 길이가 $n$인 임의의 수열 $S$가 주어질 때 이 수열이 좋은 수열인지 다음과 같이 판단할 수 있다

`-` 임의의 $i,j$에 대해 $S_i \sim S_j$와 $S_{j + 1}\sim S_{2j - i + 1}$이 동일하면 나쁜 수열이다

`-` 가능한 $i,j$ 쌍의 수는 $O\left(n^2\right)$이며 $i,j$가 주어졌을 때 부분 수열이 동일한지 판단하는 건 $O(n)$이다

`-` 나쁜 수열이 아니면 좋은 수열이므로 $O\left(n^3\right)$에 좋은 수열 여부를 확인할 수 있다

`-` 길이가 $N$일 때 가능한 수열의 경우의 수는 $O\left(3^N\right)$이며 이들 각각이 좋은 수열인지 판단해야 하니 $O\left(N^3 3^N\right)$이다

`-` $N$이 최대 $80$이므로 제한 시간 안에 동작할 수 없지만 백트래킹을 통해 나쁜 수열을 걸러내며 좋은 수열을 확장시켜 나간다면 경우의 수가 많이 줄어들므로 빠르게 실행될 것이다

`-` 참고로 가능한 좋은 수열 중 가장 작은 수를 나타내는 것만 계산하면 된다. 따라서 $1, 2, 3$ 순으로 좋은 수열을 확장시키자. 그럼 가장 먼저 완성되는 길이가 $N$인 좋은 수열이 정답이 된다

In [21]:
def backtracking(sequence, n_goal):
    global ANSWER
    if ANSWER is not None:
        return
    if len(sequence) == n_goal:
        ANSWER = sequence[:]
        return
    for x in [1, 2, 3]:
        sequence.append(x)
        if is_good_sequence(sequence):  
            backtracking(sequence, n_goal)
        sequence.pop()


def is_good_sequence(sequence):
    n = len(sequence)
    for j in range(n):
        for i in range(j + 1):
            i2 = j + 1
            j2 = 2 * j - i + 1
            if j2 >= n:
                continue
            s1 = sequence[i : j + 1]
            s2 = sequence[i2 : j2 + 1]
            if s1 == s2:
                return False
    return True


def solution():
    global ANSWER
    N = int(input())
    ANSWER = None
    backtracking([], N)
    print("".join(map(str, ANSWER)))


solution()

# input
# 11

 11


12131231321


## N Queen

- 문제 출처: [정올 1889번](https://jungol.co.kr/problem/1889)

`-` 각 칸마다 퀸의 이동 영향을 받는지 기록하는지 대신 열, 우상대각선, 우하대각선의 인덱스에 정보를 저장하자

`-` 러프하게 대각선을 고려하지 않는 경우엔 시간 복잡도가 $O(n!)$이지만 대각선까지 고려해 가지치기를 수행하므로 충분히 빠르게 실행된다

In [3]:
def dfs(row, col_used, slash_used, backslash_used):
    n = len(col_used)
    if row == n:
        return 1
    count = 0
    for col in range(n):
        s = row + col
        bs = row + n - 1 - col
        if col_used[col] or slash_used[s] or backslash_used[bs]:
            continue
        col_used[col] = slash_used[s] = backslash_used[bs] = True
        count += dfs(row + 1, col_used, slash_used, backslash_used)
        col_used[col] = slash_used[s] = backslash_used[bs] = False
    return count


def solution():
    N = int(input())
    col_used = [False] * N
    slash_used = [False] * (2 * N - 1)
    backslash_used = [False] * (2 * N - 1)
    answer = dfs(0, col_used, slash_used, backslash_used)
    print(answer)


solution()

# input
# 10

 10


724


## Dessert

- 문제 출처: [정올 1180번](https://jungol.co.kr/problem/1180)

`-` 브루트 포스로 구현하면 $O\left(N 3^{N}\right)$이다. $N$이 최대 $15$라 애매하다

`-` $+$ 또는 $-$가 입력되면 여태까지의 수식 결과가 하나로 결정됨에 집중하자. $+,-$가 들어오면 수식 결과를 갱신하고 추적하는 수 $t$를 각각 $x,-x$로 초기화하자

`-` 반대로 $.$이 입력되면 추적하는 수에 $10^{d(x)}$을 곱한 뒤 $(-1)^{I(t < 0)}\cdot x$를 더하자. 단, $d(x)$는 $x$의 자릿수이다

`-` 초기에 $t=1$이며 $x$는 $2$부터 $N$까지이다. 또한, 정답 역추적을 위해 수식 기록을 가지고 있어야 한다

`-` 결과적으로 백트래킹을 통해 이전 정보를 재사용하여 시간 복잡도를 $O\left(3^{N}\right)$로 만들 수 있다

In [39]:
def dfs(r, t, x, n, history, formulas):
    if x > n:
        if r + t == 0:
            if len(formulas) < 20:
                formulas.append(history[:])
            return 1
        return 0
    count = 0
    d = len(str(x))
    it = zip("+-.", [t, t, 0], [0, 0, 10**d], [x, -x, x * (-1)**(t < 0)])
    for op, dr, mul, add in it:
        history.extend([op, x])
        count += dfs(r + dr, t * mul + add, x + 1, n, history, formulas)
        del history[-2:]
    return count


def solution():
    N = int(input())
    formulas = []
    count = dfs(0, 1, 2, N, [1], formulas)
    for f in formulas:
        print(*f)
    print(count)


solution()

# input
# 7

 7


1 + 2 - 3 + 4 - 5 - 6 + 7
1 + 2 - 3 - 4 + 5 + 6 - 7
1 - 2 + 3 + 4 - 5 + 6 - 7
1 - 2 - 3 - 4 - 5 + 6 + 7
1 - 2 . 3 + 4 + 5 + 6 + 7
1 - 2 . 3 - 4 . 5 + 6 . 7
6


## 벽장문의 이동

- 문제 출처: [정올 1409번](https://jungol.co.kr/problem/1409)

`-` 사용해야 될 벽장이 열려있으면 그냥 사용하면 된다. 닫혀있다면 왼쪽 또는 오른쪽으로 문을 밀자. 그러면 닫힌 벽장은 열리고 열린 벽장 하나는 닫힌다. 벽장의 상태는 비트마스크로 관리하자

`-` 열린 벽장은 $2$개뿐이니 most significant set bit와 least significant set bit를 구하면 되고 현재 사용할 벽장과의 위치 관계는 최대 $3$개이다. 각 상황에 맞춰 처리하면 된다

`-` 초기 벽장의 상태는 하나이며 매 차례 비트마스크의 변화는 최대 $2$번이니 백트래킹으로 구현하면 총 $O\left(2^N\right)$이다

In [5]:
def dfs(i, n_moves, opened, order):
    global MIN
    m = len(order)
    if i >= m:
        MIN = min(n_moves, MIN)
        return
    x = order[i]
    xb = 1 << x
    if opened & xb:
        dfs(i + 1, n_moves, opened, order)
        return
    msb = 1 << (opened.bit_length() - 1)
    lsb = opened & -opened
    dm = abs(msb.bit_length() - 1 - x)
    dl = abs(x - lsb.bit_length() + 1)
    if xb > msb:
        dfs(i + 1, n_moves + dm, (opened ^ msb) | xb, order)
    elif xb < lsb:
        dfs(i + 1, n_moves + dl, (opened ^ lsb) | xb, order)
    else:
        dfs(i + 1, n_moves + dm, (opened ^ msb) | xb, order)
        dfs(i + 1, n_moves + dl, (opened ^ lsb) | xb, order)


def solution():
    global MIN
    N = int(input())
    a, b = map(lambda x: int(x) - 1, input().split())
    M = int(input())
    order = [int(input()) - 1 for _ in range(M)]
    MIN = float("inf")
    dfs(0, 0, (1 << a) | (1 << b), order)
    print(MIN)


solution()

# input
# 4
# 1 2
# 2
# 3
# 4

 4
 1 2
 2
 3
 4


2


`-` 그냥 비트마스크 안 쓰고 배열로 관리하는 게 훨씬 낫다. 핵심은 열린 두 문의 위치를 추적하는 것 (굳이 비트 연산 하면서 온라인으로 구할 필요가 없다)

`-` 근데 이렇게 한 이유가 있음. 문을 밀었을 때 상태가 변하는 일련의 과정에 너무 집중해서 결과를 제대로 써먹을 생각을 못했다. 결과를 보면 그냥 교환이랑 다를 바가 없다...

`-` 근데 이러면 배열조차 쓸 필요가 없다. 어차피 열린 문은 $2$개이므로 이것만 알고 있으면 됨

## 비숍

- 문제 출처: [정올 1662번](https://jungol.co.kr/problem/1662)

`-` 체스판에서 우상 방향의 대각선 수는 $2N - 1$이다. 각 대각선마다 최대 $1$개의 비숍만 배치할 수 있으므로 고려할 경우의 수는 $N!\cdot (N-1)!$이다. $N$이 최대 $10$이므로 가지치기를 잘 수행해도 TLE이다

`-` 백준에서 풀었을 땐 제한 시간도 더 많고 휴리스틱을 잘 섞어서 간신히 통과했는데 여기선 불가능하다

`-` 비숍의 이동 특성상 밝은색 비숍과 어두운색 비숍으로 나눠진다. 둘은 독립이므로 두 경우의 최댓값을 더해주면 전체 경우의 최댓값이 된다. 따라서 시간 복잡도는 최악의 경우에도 $O(N!)$이다

In [111]:
def maximize_bishop(i, grid):
    global MAX
    MAX = 0
    dfs(i, 0, [False] * (2 * len(grid) - 1), grid)
    return MAX


def dfs(i, k, backslash_used, grid):
    global MAX
    if MAX < k:
        MAX = k
    m = len(backslash_used)
    if i >= m:
        return
    n = (m + 1) // 2
    q = abs(n - 1 - i) // 2
    order = range(abs(n - 1 - i) % 2, m, 2)
    order = order[q : len(order) - q]
    is_placed = False
    for j in order:
        r = (i + j - n + 1) // 2
        c = (i - j + n - 1) // 2
        if backslash_used[j] or grid[r][c] == 0:
            continue
        is_placed = True
        backslash_used[j] = True
        dfs(i + 2, k + 1, backslash_used, grid)
        backslash_used[j] = False
    if not is_placed:
        dfs(i + 2, k, backslash_used, grid)


def solution():
    N = int(input())
    grid = [list(map(int, input().split())) for _ in range(N)]
    answer = maximize_bishop(0, grid) + maximize_bishop(1, grid)
    print(answer)


solution()

# input
# 3
# 1 1 1
# 0 0 1
# 0 0 1

 3
 1 1 1
 0 0 1
 0 0 1


3


`-` 구현하는 데 너무 오래 걸린다. 나는 인덱스 처리하는 게 필요한 문제에 약하다...

`-` 처음엔 대각선에 비숍을 일부러 놓지 않는 경우도 고려했다 (미래에 더 나은 선택을 위해)

`-` 맞히고 다른 사람 코드 보면서 생각해보니 이럴 필요는 없다. 아꼈다가 미래에 쓰겠다는 건데 어차피 각 대각선마다 기회는 $1$번이라 지금 쓰나 미래에 쓰나 차이가 없다